<a href="https://colab.research.google.com/github/Mahendra2409/PyBlender/blob/main/Colab_Script/.test_within_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<a href="https://drive.google.com/drive/folders/1sj-RqD5HRypGx-ZLqXpvyzY1CN84qJu-?usp=drive_link" target="_parent"><img src="https://img.shields.io/badge/PyBlender_Render_Farm-blue?logo=googledrive&logoColor=white" alt="PyBlender_Render_Farm"/></a>

## 1.Install Dependencies

In [ ]:
# @title
# 1. Download and extract Blender 4.0.2
!wget -nc https://download.blender.org/release/Blender4.0/blender-4.0.2-linux-x64.tar.xz
!tar -xf blender-4.0.2-linux-x64.tar.xz

# 2. Install dependencies into Blender 4.0's bundled Python
!./blender-4.0.2-linux-x64/4.0/python/bin/python3.10 -m ensurepip
!./blender-4.0.2-linux-x64/4.0/python/bin/python3.10 -m pip install scipy matplotlib numpy blendertoolbox

In [ ]:
# @title
# 3. Download BlenderToolbox
!git clone https://github.com/HTDerekLiu/BlenderToolbox.git
# Move the actual toolbox folder into your main /content/ directory so your script can 'see' it

!mv BlenderToolbox/blendertoolbox /content/
!./blender-4.0.2-linux-x64/4.0/python/bin/python3.10 -m pip install blendertoolbox

In [ ]:
#@title Upload Point Cloud File
import os
import shutil
from google.colab import files

# 1. Define the directory where you want the files to go
# You can change this path to match your project needs
target_directory = '/content/uploaded_pointclouds'

# 2. Create the directory if it doesn't already exist
os.makedirs(target_directory, exist_ok=True)

print(f"Select files to upload. They will be saved to: {target_directory}\n")

# 3. Open the file selection pop-up
# This saves the files temporarily to the current working directory (/content/)
uploaded_files = files.upload()

# 4. Move each uploaded file into your target directory
for filename in uploaded_files.keys():
    # Define source and destination paths
    source_path = filename
    destination_path = os.path.join(target_directory, filename)

    # Move the file (overwrites if a file with the same name already exists in the target folder)
    shutil.move(source_path, destination_path)

print(f"\n✅ Success! All files have been moved to {target_directory}")

In [ ]:
#@title 1. Set Configuration Here
%%writefile config.py

# ==========================================
# 1. MAIN CONFIGURATION
# ==========================================
CONFIG = {
    # --- Output Controls ---
    "SAVE_BLEND_FILE": True,
    "FORCE_OVERWRITE": True,

    # --- Paths ---
    "SOURCE_FOLDER_PATH": "/content/uploaded_pointclouds",
    "OUTPUT_FOLDER_PATH": "/content/rendered_outputs",
    "GT_FILENAME": "ellipsoid.xyz",
    "COLORMAP": "viridis",

    # --- Object Transforms ---
    "OBJ_LOCATION": (0.007266, 0.137527, 0.187947),
    "OBJ_ROTATION": (562.435, 17.477, -329.939),
    "OBJ_SCALE": (2.0, 2.0, 2.0),

    # --- Point Cloud Settings ---
    "PT_SIZE": 0.004,
    "PT_COLOR": [0.5, 1.0, 1.0, 0.0, 0.0],



    # --- Blender Render Settings ---
    "IMG_RES_X": 1000,
    "IMG_RES_Y": 1000,
    "NUM_SAMPLES": 100,
    "EXPOSURE": 1.5,

    # --- Camera Settings ---
    "CAM_LOCATION": (0.141656, 2.50472, 1.37419),
    "LOOK_AT": (0, 0, 0),
    "FOCAL_LENGTH": 45,

    # --- Light Settings ---
    "LIGHT_ANGLE": (6, -30, -155),
    "LIGHT_STRENGTH": 2,
    "SHADOW_SOFTNESS": 0.3,
    "AMBIENT_COLOR": (0.1, 0.1, 0.1, 1),
    "SHADOW_THRESHOLD": 0.05
}

# ==========================================
# 2. GROUND TRUTH OVERRIDES
# ==========================================
GT_OVERRIDES = {
    # Example: Uncomment to apply only to GT
    # "PT_SIZE": 0.012,
}

In [ ]:
#@title 2. Render Pipeline
%%writefile render.py

import os
import sys
import shutil
import numpy as np
from scipy.spatial import cKDTree

# ---> THE FIX: Tell Blender exactly where to look for config.py <---
# Colab saves files to '/content/', so we add it to Blender's Python path.
if '/content' not in sys.path:
    sys.path.append('/content')

from config import CONFIG, GT_OVERRIDES

# --- THE FIX FOR COLAB/MATPLOTLIB ---
os.environ['MPLBACKEND'] = 'Agg'
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
# ------------------------------------

import bpy
import blendertoolbox as bt

# ==========================================
# DERIVED PATHS
# ==========================================
LOCAL_SOURCE_DIR = CONFIG["SOURCE_FOLDER_PATH"]
OUTPUT_DIR = CONFIG["OUTPUT_FOLDER_PATH"]
LOCAL_GROUND_TRUTH_PATH = os.path.join(LOCAL_SOURCE_DIR, CONFIG["GT_FILENAME"])


# ==========================================
# PIPELINE FUNCTIONS
# ==========================================
def setup_colab_environment():
    print("--- Setting up Environment ---")

    if not os.path.exists(LOCAL_SOURCE_DIR):
        raise FileNotFoundError(f"Source directory not found: {LOCAL_SOURCE_DIR}. Please check your uploaded folder path.")

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"Reading point clouds directly from: {LOCAL_SOURCE_DIR}")
    print(f"Outputs will be saved to: {OUTPUT_DIR}\n")


def render_point_clouds():
    print("--- Starting Render Process ---")

    if not os.path.exists(LOCAL_GROUND_TRUTH_PATH):
        raise FileNotFoundError(f"Ground truth file not found: {LOCAL_GROUND_TRUTH_PATH}")

    ground_truth_points = np.loadtxt(LOCAL_GROUND_TRUTH_PATH)
    tree = cKDTree(ground_truth_points)

    for filename in os.listdir(LOCAL_SOURCE_DIR):
        if not filename.endswith(".xyz"):
            continue

        is_gt = (filename == CONFIG["GT_FILENAME"])

        active_cfg = CONFIG.copy()
        if is_gt:
            active_cfg.update(GT_OVERRIDES)

        render_output = os.path.join(OUTPUT_DIR, f"{os.path.splitext(filename)[0]}.png")
        if os.path.exists(render_output) and not active_cfg["FORCE_OVERWRITE"]:
            print(f"Output already exists: {filename}. Skipping...")
            continue

        print(f"Processing: {filename} {'(GROUND TRUTH MODE)' if is_gt else ''}")

        bt.blenderInit(
            active_cfg["IMG_RES_X"],
            active_cfg["IMG_RES_Y"],
            active_cfg["NUM_SAMPLES"],
            active_cfg["EXPOSURE"]
        )

        current_points = ground_truth_points if is_gt else np.loadtxt(os.path.join(LOCAL_SOURCE_DIR, filename))

        if is_gt:
            colormap = plt.get_cmap(active_cfg["COLORMAP"])
            min_color = colormap(0.0)[:3]
            colors = np.tile(min_color, (current_points.shape[0], 1))
        else:
            distances, _ = tree.query(current_points)
            max_distance = np.percentile(distances, 99)
            normalized_distances = np.clip(distances / max_distance, 0, 1)
            normalized_distances = np.log1p(normalized_distances) / np.log1p(1)
            colormap = plt.get_cmap(active_cfg["COLORMAP"])
            colors = colormap(normalized_distances)[:, :3]

        mesh = bt.readNumpyPoints(
            current_points,
            active_cfg["OBJ_LOCATION"],
            active_cfg["OBJ_ROTATION"],
            active_cfg["OBJ_SCALE"]
        )
        mesh = bt.setPointColors(mesh, colors)

        ptColor = bt.colorObj(active_cfg["PT_COLOR"], 0.5, 1.0, 1.0, 0.0, 0.0)
        bt.setMat_pointCloudColored(mesh, ptColor, active_cfg["PT_SIZE"])

        cam = bt.setCamera(active_cfg["CAM_LOCATION"], active_cfg["LOOK_AT"], active_cfg["FOCAL_LENGTH"])
        sun = bt.setLight_sun(active_cfg["LIGHT_ANGLE"], active_cfg["LIGHT_STRENGTH"], active_cfg["SHADOW_SOFTNESS"])
        bt.setLight_ambient(color=active_cfg["AMBIENT_COLOR"])
        bt.shadowThreshold(alphaThreshold=active_cfg["SHADOW_THRESHOLD"], interpolationMode='CARDINAL')

        bpy.context.scene.use_nodes = True
        tree_nodes = bpy.context.scene.node_tree
        tree_nodes.nodes.clear()

        render_layers = tree_nodes.nodes.new('CompositorNodeRLayers')
        denoise_node = tree_nodes.nodes.new(type='CompositorNodeDenoise')
        composite = tree_nodes.nodes.new('CompositorNodeComposite')
        viewer = tree_nodes.nodes.new('CompositorNodeViewer')

        render_layers.location = (-300, 0)
        denoise_node.location = (0, 0)
        composite.location = (300, 0)
        viewer.location = (300, -200)

        tree_nodes.links.new(render_layers.outputs['Image'], denoise_node.inputs['Image'])
        tree_nodes.links.new(denoise_node.outputs['Image'], composite.inputs['Image'])
        tree_nodes.links.new(denoise_node.outputs['Image'], viewer.inputs['Image'])

        colormap_path = os.path.join(OUTPUT_DIR, f"{active_cfg['COLORMAP']}_colormap.png")
        if not os.path.exists(colormap_path):
            gradient = np.linspace(0, 1, 256).reshape(1, -1)
            gradient = np.vstack([gradient] * 50)
            plt.figure(figsize=(6, 1))
            plt.imshow(gradient, aspect="auto", cmap=active_cfg["COLORMAP"])
            plt.axis("off")
            plt.savefig(colormap_path, bbox_inches="tight", pad_inches=0, dpi=300)
            plt.close()
            print(f"Colormap saved at {colormap_path}")

        if active_cfg["SAVE_BLEND_FILE"]:
            blend_file_name = f"{os.path.splitext(filename)[0]}.blend"
            blend_file_path = os.path.join(OUTPUT_DIR, blend_file_name)
            bpy.ops.wm.save_mainfile(filepath=blend_file_path)
            print(f"Blend file saved at {blend_file_path}")

        bt.renderImage(render_output, cam)
        print(f"Render complete for {filename}\n")


if __name__ == "__main__":
    setup_colab_environment()
    render_point_clouds()

In [ ]:
#@title 2.3 Start Rendering Render

!./blender-4.0.2-linux-x64/blender -b -P render.py

In [ ]:
#@title Extra: 3D PointCloud Visualizer
import numpy as np
import plotly.graph_objs as go

# 1. PATH TO YOUR POINT CLOUD
# Point this to any .xyz file you want to inspect (either on Drive or Local Colab storage)
PC_PATH = "/content/local_data/ccylinder_scaled_PC_new/AD_ccylinder_gaussian_3.5.xyz"

def view_point_cloud(filepath, max_points=100000):
    print(f"Loading {filepath}...")
    try:
        points = np.loadtxt(filepath)
    except Exception as e:
        print(f"Error loading file: {e}")
        return

    # 2. OPTIMIZATION
    # Browsers will crash if you try to render millions of points in Plotly.
    # If the point cloud is huge, we randomly sample it down to keep it interactive.
    if len(points) > max_points:
        print(f"Subsampling from {len(points)} to {max_points} points for smooth viewing...")
        indices = np.random.choice(len(points), max_points, replace=False)
        points = points[indices]

    x, y, z = points[:, 0], points[:, 1], points[:, 2]

    # Calculate center bounding box for reference
    center_x = (np.max(x) + np.min(x)) / 2
    center_y = (np.max(y) + np.min(y)) / 2
    center_z = (np.max(z) + np.min(z)) / 2
    print(f"\n--- Point Cloud Stats ---")
    print(f"X range: {np.min(x):.2f} to {np.max(x):.2f} (Center: {center_x:.2f})")
    print(f"Y range: {np.min(y):.2f} to {np.max(y):.2f} (Center: {center_y:.2f})")
    print(f"Z range: {np.min(z):.2f} to {np.max(z):.2f} (Center: {center_z:.2f})")

    # 3. BUILD THE INTERACTIVE PLOT
    print("\nGenerating interactive 3D viewer...")
    fig = go.Figure(data=[go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=dict(
            size=1.5,
            color=z,                 # Color points by height to give depth
            colorscale='Viridis',
            opacity=0.8
        )
    )])

    # Lock the aspect ratio so the object doesn't stretch weirdly
    fig.update_layout(
        scene=dict(
            aspectmode='data',
            xaxis_title='X Axis',
            yaxis_title='Y Axis',
            zaxis_title='Z Axis'
        ),
        margin=dict(l=0, r=0, b=0, t=0),
        height=700
    )

    fig.show()

# Run the viewer
view_point_cloud(PC_PATH)